In [12]:
include("SystemResponses.jl")
include("SymolicDerivative.jl")

using DifferentialEquations, SymbolicRegression, Logging
using .SymbolicDerivativeModule

# Define operators
operators = OperatorEnum(
    binary_operators=(+, *, -, /),
    unary_operators=(cos, exp)
)

variable_names = ["x1", "x2", "x3"]

# Define variables (feature 2 is x2, feature 3 is x3)
x2 = Expression(Node{Float64}(feature=2); operators, variable_names)
x3 = Expression(Node{Float64}(feature=3); operators, variable_names)

# Construct the expression: -2 * x2 + 0.5 * x3
expression = -2.0 * x2 + 0.5 * x3
result = (x2 / -0.24189631519383992) + x3

# Test evaluation
X = randn(Float64, 3, 100)
output = expression(X)

println("Expression: ", expression)
println("Result: ", result)

Expression: (-2.0 * x2) + (0.5 * x3)
Result: (x2 / -0.24189631519383992) + x3


In [2]:
t = 0:0.1:10
c = sin.(t) # control input
x = SystemResponsesModule.linear_controlled_system_response(t, c; a=2.0, b=0.5, error_std=0.0);
t_x_c = vcat(t', x', c')

dataset = Dataset(t_x_c, x);

In [13]:
tree_node = expression.tree

options = Options(binary_operators=(+, *, -, /), unary_operators=(cos, exp))

# Create the population member
member = PopMember(
    dataset,
    expression,
    options;
    deterministic=true
)
result_member = PopMember(
    dataset,
    result,
    options,
    deterministic=true
)

println("Raw Node type: ", typeof(tree_node))
println("PopMember: ", member)

Raw Node type: Node{Float64, 2}
PopMember: PopMember(tree = ((-2.0 * x2) + (0.5 * x3)), loss = 0.05134834190530717, cost = 0.05134834190530717)


In [4]:
options_deriv = SymbolicRegression.Options(;
    binary_operators=[+, *, /, -], 
    unary_operators=[cos, sin, exp], # Added sin
    seed=42,
    output_directory="output/deriv_search" # Save results to this file
)

function derivative_ode_func(u, p_tree, t)
    c_val = sin(t) 
    
    # Construct input vector for the tree: [t, x, c]
    # u corresponds to x (the state)
    input_features = [t, u, c_val] 
    
    # Reshape for eval_tree_array (expects matrix where columns are samples)
    input_matrix = reshape(input_features, :, 1) 
    
    try
        val, success = eval_tree_array(p_tree, input_matrix, options_deriv)
        if success && isfinite(val[1])
            return val[1]
        else
            return 0.0
        end
    catch e
        if isa(e, DomainError)
            return 0.0
        else
            rethrow(e)
        end
    end
end

derivative_ode_func (generic function with 1 method)

In [5]:
function l2_loss_ode(tree, dataset, options)
    t = dataset.X[1, :]
    x = dataset.y
    tspan = (t[1], t[end])

    ode_problem = ODEProblem(derivative_ode_func, x[1], tspan, tree)
    solution = solve(
        ode_problem, 
        AutoTsit5(Rosenbrock23()),
        maxiters=5000,
        saveat=t, 
        abstol=1e-3,
        reltol=1e-3,
        verbose=0,
        # warn_dtmin=false,
        # warn_on_unstable=false
    )

    if SciMLBase.successful_retcode(solution) && length(solution.u) == length(x)
        loss = sum((solution.u .- x).^2) / length(x)
    else
        loss = Inf
    end
    
    return loss
end

l2_loss_ode (generic function with 1 method)

In [6]:

function symolic_integration(t_x_c, x, dominating_derivatives, options)
    guess_trees = [entry.tree for entry in dominating_derivatives]

    with_logger(NullLogger()) do
        println("--- Starting Search Using Integration Loss ---")
        hall_of_fame_ode = equation_search(
            t_x_c, x; 
            options=options,
            niterations = 5, # WARNING: This is very slow. 5 is just for a quick test.
                            # A real search might need 50+.
            parallelism = :multithreading,
            guesses = guess_trees
        )
        dominating_ode = calculate_pareto_frontier(hall_of_fame_ode)
    return dominating_ode
    end
end


symolic_integration (generic function with 1 method)

In [14]:
l2_loss_ode(result_member.tree, dataset, options_deriv)

1.4578046488050405e7

In [9]:
dominating_derivatives = SymbolicDerivativeModule.symbolic_derivative(t, x, c);


nfo: Started!
















































































Evolving for 10 iterations... 100%|██████████████████████| Time: 0:00:07





















Evolving for 10 iterations... 100%|██████████████████████| Time: 0:00:07


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.273e-02  0.000e+00  y = -0.0030955
3           1.808e-02  1.144e-01  y = x₃ * 0.09916
4           4.916e-03  1.302e+00  y = cos(x₁) * 0.1841
5           4.915e-03  2.637e-04  y = sin(cos(x₁)) * 0.20963
6           8.460e-04  1.759e+00  y = sin(x₁ + 1.0977) * 0.20348
7           7.069e-05  2.482e+00  y = (x₃ + (x₂ / -0.25572)) * 0.48166
8           6.957e-05  1.594e-02  y = (x₃ * 0.48168) + (sin(x₂) / -0.52743)
12          6.955e-05  7.595e-05  y = ((x₂ * -0.20578) + (x₃ * 0.48165)) + sin(x₂ / -0.58479...
                                      )
14          6.925e-05  2.171e-03  y = ((x₃ + (x₂ / -0.39687)) + sin((x₂ / -0.71125) + -0.001...
                                      696)) * 0.48164
16          4.334e-05  2.344e-01  y = (0.3275 / ((0.33556 / ((x₁ * 0.67628) + 0.053066)) + 1...
                                      .3911)) *

┌ Info: Final population:
└ @ SymbolicRegression /home/fidelius/SymbolicRegression.jl/src/SymbolicRegression.jl:1223


  - outputs/20251205_133115_96dFrM/hall_of_fame.csv


┌ Info: Results saved to:
└ @ SymbolicRegression /home/fidelius/SymbolicRegression.jl/src/SymbolicRegression.jl:1246


In [10]:
options_ode = SymbolicRegression.Options(;
    binary_operators=[+, *, /, -], 
    unary_operators=[cos, sin, exp],
    loss_function=l2_loss_ode,
    seed = 42,
    output_directory="output/ode_search", # Save results to this file
)

symolic_integration(t_x_c, x, dominating_derivatives, options_ode)

--- Starting Search Using Integration Loss ---


2mEvolving for 5 iterations...   1%|▎                      |  ETA: 0:59:05














































Evolving for 5 iterations... 100%|███████████████████████| Time: 0:02:21


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.300e-02  0.000e+00  y = 0.0069708
3           1.891e-02  9.786e-02  y = 0.0019489 / x₃
4           5.076e-03  1.315e+00  y = cos(x₁) * 0.20306
5           1.361e-03  1.316e+00  y = (x₂ / -0.2419) + x₃
6           9.662e-04  3.426e-01  y = 0.20197 * cos(x₁ / 1.0753)
7           1.997e-06  6.182e+00  y = (x₃ + (x₂ / -0.25586)) * 0.49757
8           1.867e-06  6.748e-02  y = (sin(x₂) / -0.51088) + (x₃ * 0.49762)
12          1.804e-06  8.487e-03  y = (x₂ * -0.27241) + (sin(x₂ / -0.58688) + (0.49756 * x₃)...
                                      )
14          1.729e-06  2.133e-02  y = 0.49773 * (((x₂ / -0.39742) + x₃) + sin(-0.0018003 + (...
                                      x₂ / -0.71118)))
19          2.054e-07  4.261e-01  y = (x₂ / -0.46606) + (0.53509 * ((cos(x₁ - -0.19149) * (s...
                                      in(0.0307

10-element Vector{PopMember{Float64, Float64, Expression{Float64, Node{Float64, 2}, @NamedTuple{operators::OperatorEnum{Tuple{Tuple{typeof(cos), typeof(sin), typeof(exp)}, Tuple{typeof(+), typeof(*), typeof(/), typeof(-)}}}, variable_names::Vector{String}}}}}:
 PopMember(tree = (0.006970787970549503), loss = 0.022996085812517292, cost = 0.9338923881903081)
 PopMember(tree = (0.0019489407166138066 / x3), loss = 0.018908303265707402, cost = 0.7678837449730798)
 PopMember(tree = (cos(x1) * 0.2030643808253817), loss = 0.0050758041398656, cost = 0.20613311712313706)
 PopMember(tree = ((x2 / -0.24189631519383992) + x3), loss = 0.0013610326592430333, cost = 0.055272799506323654)
 PopMember(tree = (0.2019684188841977 * cos(x1 / 1.0752759601073087)), loss = 0.0009662050262208282, cost = 0.03923848287814655)
 PopMember(tree = ((x3 + (x2 / -0.2558640805277155)) * 0.49756589573245386), loss = 1.9969039738601537e-6, cost = 8.109612376380469e-5)
 PopMember(tree = ((sin(x2) / -0.5108813617277871) + (